# Asset Deep Dive
Full analysis of a single symbol: OHLCV chart, rolling metrics, candle statistics.

Change `SYMBOL` below to analyse any tracked asset.

In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import clickhouse_connect
from dotenv import load_dotenv

load_dotenv()

# ---- CONFIG ----
SYMBOL   = 'BTC/USDT'
EXCHANGE = 'binance'
DAYS     = 30
# ----------------

ch = clickhouse_connect.get_client(
    host=os.getenv('CLICKHOUSE_HOST', 'clickhouse'),
    port=int(os.getenv('CLICKHOUSE_HTTP_PORT', '8123')),
    database=os.getenv('CLICKHOUSE_DB', 'crypto'),
    username=os.getenv('CLICKHOUSE_USER', 'crypto_user'),
    password=os.getenv('CLICKHOUSE_PASSWORD', ''),
)
print(f'Analysing {SYMBOL} on {EXCHANGE} — last {DAYS} days')

## 1. Candlestick Chart (1h aggregated)

In [ ]:
ohlcv = ch.query_df(f"""
SELECT
    toStartOfHour(open_time) AS hour,
    argMin(open,  open_time) AS open,
    max(high)                AS high,
    min(low)                 AS low,
    argMax(close, open_time) AS close,
    sum(volume)              AS volume,
    sum(quote_volume)        AS quote_volume
FROM crypto.fact_candles
WHERE symbol = '{SYMBOL}' AND exchange = '{EXCHANGE}'
  AND interval = '1m'
  AND open_time >= now() - INTERVAL {DAYS} DAY
GROUP BY hour
ORDER BY hour
""")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.75, 0.25],
                    subplot_titles=[f'{SYMBOL} Price (1h)', 'Volume'])

fig.add_trace(go.Candlestick(
    x=ohlcv['hour'], open=ohlcv['open'], high=ohlcv['high'],
    low=ohlcv['low'], close=ohlcv['close'], name='OHLC'
), row=1, col=1)

colors = ['green' if c >= o else 'red' for c, o in zip(ohlcv['close'], ohlcv['open'])]
fig.add_trace(go.Bar(
    x=ohlcv['hour'], y=ohlcv['quote_volume'], marker_color=colors, name='Vol (USDT)'
), row=2, col=1)

fig.update_layout(height=700, title=f'{SYMBOL} — last {DAYS} days', xaxis_rangeslider_visible=False)
fig.show()

## 2. Rolling Volatility Over Time

In [ ]:
vol_ts = ch.query_df(f"""
SELECT
    window_start,
    round(realized_vol_7d  * 100, 2) AS vol_7d_pct,
    round(realized_vol_30d * 100, 2) AS vol_30d_pct,
    round(avg_true_range,          4) AS atr
FROM crypto.mart_volatility
WHERE symbol = '{SYMBOL}' AND exchange = '{EXCHANGE}'
ORDER BY window_start
""")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Realized Volatility (annualized %)', 'ATR (14d)'])
fig.add_trace(go.Scatter(x=vol_ts['window_start'], y=vol_ts['vol_7d_pct'],
                          name='Vol 7d', line=dict(color='orange')), row=1, col=1)
fig.add_trace(go.Scatter(x=vol_ts['window_start'], y=vol_ts['vol_30d_pct'],
                          name='Vol 30d', line=dict(color='blue', dash='dot')), row=1, col=1)
fig.add_trace(go.Scatter(x=vol_ts['window_start'], y=vol_ts['atr'],
                          name='ATR', line=dict(color='purple')), row=2, col=1)
fig.update_layout(height=500, title=f'{SYMBOL} Volatility Timeline')
fig.show()

## 3. Candle Distribution Stats

In [ ]:
stats = ch.query_df(f"""
SELECT
    count()                                      AS total_candles,
    round(avg(price_change_pct),    4)           AS avg_pct_change,
    round(stddevPop(price_change_pct), 4)        AS std_pct_change,
    round(avg(candle_range),        4)           AS avg_range,
    round(max(candle_range),        4)           AS max_range,
    round(sum(is_bullish) / count() * 100, 1)   AS pct_bullish,
    round(avg(volume),              2)           AS avg_volume,
    round(max(volume),              2)           AS max_volume
FROM crypto.fact_candles
WHERE symbol = '{SYMBOL}' AND exchange = '{EXCHANGE}'
  AND interval = '1m'
  AND open_time >= now() - INTERVAL {DAYS} DAY
""")
display(stats.T.rename(columns={0: 'value'}))

In [ ]:
# Distribution of 1m returns
returns = ch.query_df(f"""
SELECT price_change_pct
FROM crypto.fact_candles
WHERE symbol = '{SYMBOL}' AND exchange = '{EXCHANGE}'
  AND interval = '1m'
  AND open_time >= now() - INTERVAL {DAYS} DAY
  AND abs(price_change_pct) < 5
""")

fig = px.histogram(returns, x='price_change_pct', nbins=100,
                   title=f'{SYMBOL} — Distribution of 1m Returns (last {DAYS}d)',
                   labels={'price_change_pct': 'Return (%)', 'count': 'Candles'},
                   color_discrete_sequence=['steelblue'])
fig.add_vline(x=0, line_dash='dash', line_color='red')
fig.show()

## 4. Intraday Volume Profile (avg by hour of day)

In [ ]:
vp = ch.query_df(f"""
SELECT
    toHour(open_time) AS hour_of_day,
    round(avg(sum_vol), 0) AS avg_hourly_vol
FROM (
    SELECT toHour(open_time) AS hour_of_day,
           toDate(open_time) AS d,
           sum(quote_volume) AS sum_vol
    FROM crypto.fact_candles
    WHERE symbol='{SYMBOL}' AND exchange='{EXCHANGE}'
      AND interval='1m' AND open_time >= now() - INTERVAL {DAYS} DAY
    GROUP BY hour_of_day, d
)
GROUP BY hour_of_day
ORDER BY hour_of_day
""")

fig = px.bar(vp, x='hour_of_day', y='avg_hourly_vol',
             title=f'{SYMBOL} — Average Volume by Hour (UTC)',
             labels={'hour_of_day': 'Hour (UTC)', 'avg_hourly_vol': 'Avg USDT Volume'})
fig.show()